#**An Exploratory Data Analysis (EDA) for the FMA-Small Dataset**
**Notebook by: Elia Fel P. Bado**

#**I. Introduction**

####**Data Source & Origin**
*   Repository Link: [FMA GitHub Repository](https://github.com/mdeff/fma)
*   Citation: Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. arXiv:1612.01840.
<br>

####**Who collected the data, when, how, and for what purpose?**
* The dataset was created by Michaël Defferrard, Kirell Benzi, Pierre Vandergheynst, and Xavier Bresson in 2017 for the 18th International Society for Music Information Retrieval Conference (ISMIR). It was designed to provide an open, easily accessible audio dataset for evaluating tasks in Music Information Retrieval (MIR), aiming to overcome hurdles from the limited availability of large public audio datasets. The data was collected by querying the Free Music Archive (FMA) web API.
<br>

####**What does one row represent?**
* `tracks.csv`: One row represents a single music track.
* `features.csv`: One row represents extracted audio features (via librosa) for a track.
* `genres.csv`: One row represents a single musical genre.
* `echonest.csv`: One row represents Echonest-specific audio features for a track.
<br>

####**What does each column mean?**
* `tracks.csv`: Contains track metadata such as track ID, title, artist, genres, tags, and play counts.
* `features.csv`: Contains statistical descriptors of common audio features extracted using `librosa` (e.g., MFCCs, spectral centroid, chroma).
* `genres.csv`: Details 163 genres, including name, ID, and parent genre ID to infer hierarchy.
* `echonest.csv`: Provides Echonest audio features for a subset of 13,129 tracks.
* **NEED DATA DICTIONARY**

<br>


####**Questions to Explore**


1.   How accurately can baseline machine learning models perform genre recognition using extracted audio features vs. raw audio?
2.   How does the hierarchical genre taxonomy affect classification performance across top-level genres?
3.   Can representation learning on artist labels transfer effectively to genre classification tasks?
4.   How do Echonest audio features compare with librosa-extracted features in predicting track play counts?




In [18]:
import os


# connecting colab to gdrive, check if folder exists, and return folder size
from google.colab import drive
drive.mount('/content/drive')

#paths
base_path = '/content/drive/MyDrive/Music_Genre_Classification'
dataset_path = os.path.join(base_path, 'dataset')

raw_dataset_path = os.path.join(dataset_path, 'raw')
fma_metadata_path = os.path.join(raw_dataset_path, 'fma_metadata')
fma_small_path = os.path.join(raw_dataset_path, 'fma_small')

processed_dataset_path = os.path.join(dataset_path, 'processed')


#for fetching folder size
def get_dir_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

#to check if folders has contents
folder_exists = os.path.exists(base_path)
print(f"Music_Genre_Classification file exists: {folder_exists}")
print(f"dataset/raw/fma_metadata exists: {os.path.exists(fma_metadata_path)}")
print(f"dataset/raw/fma_small exists: {os.path.exists(fma_small_path)}")

if folder_exists:
    fma_metadata_file_size = (get_dir_size(fma_metadata_path))/(1024*1024)
    fma_small_file_size = (get_dir_size(fma_small_path))/(1024*1024)
    print(f"fma_metadata Folder Size: {fma_metadata_file_size:.2f} MB")
    print(f"fma_small Folder Size: {fma_small_file_size:.2f} MB")
else:
    print("Folder Size: N/A")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Music_Genre_Classification file exists: True
dataset/raw/fma_metadata exists: True
dataset/raw/fma_small exists: True
fma_metadata Folder Size: 1396.98 MB
fma_small Folder Size: 7606.00 MB


# **II. Data loading and first look**

Only the `tracks.csv` and `features.csv` are the useful metadata for this reserach so these two will be the only ones to be inspected.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [20]:
tracks_path = os.path.join(fma_metadata_path, 'tracks.csv')

# load the raw metadata handling the multi-index header
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])

#print the shape, sample, info, and description
print("--- Tracks Shape ---")
print(tracks.shape)

print("\n--- Tracks Random Sample (5 rows) ---")
display(tracks.sample(5, random_state=67))

print("\n--- Tracks Info ---")
tracks.info()

print("\n--- Tracks Description ---")
# Enable pandas to display all rows/columns so nothing gets truncated in the summary
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(tracks.describe(include='all').T)

--- Tracks Shape ---
(106574, 52)

--- Tracks Random Sample (5 rows) ---


album                                            \
         comments         date_created        date_released   
track_id                                                      
113262          2  2015-01-12 13:33:04                  NaN   
124022          0  2015-08-23 20:20:47  2015-08-25 00:00:00   
66876           0  2012-06-25 11:08:06                  NaN   
134564          0  2016-03-23 20:36:35  2016-03-18 00:00:00   
113665          3  2015-01-26 13:04:57                  NaN   

                                                                \
                                     engineer favorites     id   
track_id                                                         
113262                                    NaN         1  17457   
124022                            Neil Campau         0  18762   
66876     Tab, Anitek, A.U.R.C, Hernan Faraci         4  11990   
134564                                    NaN         0  20200   
113665                                    NaN         8  17509   

                                                                     \
                                                information listens   
track_id                                                              
113262                                                  NaN   91602   
124022    <p>Seven songs about the weather. <span style=...   12012   
66876                                                   NaN   23405   
134564    <p><a href="http://jeremyallingham.com" target...   13186   
113665    <p class="p1" style="margin: 0px; padding: 8px...  325595   

                                               \
                                     producer   
track_id                                        
113262                                    NaN   
124022                            Neil Campau   
66876     Tab, Anitek, A.U.R.C, Hernan Faraci   
134564                                    NaN   
113665                                    NaN   

                                                             ...  \
                                                       tags  ...   
track_id                                                     ...   
113262                                                   []  ...   
124022                                                   []  ...   
66876                                                    []  ...   
134564    ['singer-songwriter', 'rock', 'acoustic rock',...  ...   
113665    ['interiors c1964', 'existential', 'hardcore-p...  ...   

                                                      track           \
                                                information interest   
track_id                                                               
113262                                                  NaN     1352   
124022                                                  NaN     1406   
66876                                                   NaN     1380   
134564                                                  NaN     1563   
113665    <p>Steve Combs is an instrumental artist from ...     1135   

                                                                           \
         language_code                                            license   
track_id                                                                    
113262             NaN  Attribution-NonCommercial-ShareAlike 3.0 Inter...   
124022              en  Attribution-Noncommercial-Share Alike 3.0 Unit...   
66876              NaN  Attribution-NonCommercial-NoDerivatives (aka M...   
134564             NaN  Creative Commons Attribution-NonCommercial-NoD...   
113665             NaN                                  CC0 1.0 Universal   

                                                \
         listens     lyricist number publisher   
track_id                                         
113262      1158          NaN      3       NaN   
124022      1063  Neil Campau      1       NaN   
66876        988          N


--- Tracks Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 106574 entries, 2 to 155320
Data columns (total 52 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   (album, comments)            106574 non-null  int64  
 1   (album, date_created)        103045 non-null  object 
 2   (album, date_released)       70294 non-null   object 
 3   (album, engineer)            15295 non-null   object 
 4   (album, favorites)           106574 non-null  int64  
 5   (album, id)                  106574 non-null  int64  
 6   (album, information)         83149 non-null   object 
 7   (album, listens)             106574 non-null  int64  
 8   (album, producer)            18060 non-null   object 
 9   (album, tags)                106574 non-null  object 
 10  (album, title)               105549 non-null  object 
 11  (album, tracks)              106574 non-null  int64  
 12  (album, type)                100066 non-nu

count unique  \
album  comments           106574.0    NaN   
       date_created         103045  14340   
       date_released         70294   3669   
       engineer              15295    622   
       favorites          106574.0    NaN   
       id                 106574.0    NaN   
       information           83149  11075   
       listens            106574.0    NaN   
       producer              18060    760   
       tags                 106574   2388   
       title                105549  14298   
       tracks             106574.0    NaN   
       type                 100066      5   
artist active_year_begin     22711     76   
       active_year_end        5375     57   
       associated_labels     14271    754   
       bio                   71156   6084   
       comments           106574.0    NaN   
       date_created         105718  15970   
       favorites          106574.0    NaN   
       id                 106574.0    NaN   
       latitude            44544.0    NaN   
       location              70210   2329   
       longitude           44544.0    NaN   
       members               46849   4015   
       name                 106574  16294   
       related_projects      13152    752   
       tags                 106574  15660   
       website               79256   6642   
       wikipedia_page         5581    389   
set    split                106574      3   
       subset               106574      3   
track  bit_rate           106574.0    NaN   
       comments           106574.0    NaN   
       composer               3670    505   
       date_created         106574  86169   
       date_recorded          6159    773   
       duration           106574.0    NaN   
       favorites          106574.0    NaN   
       genre_top             49598     16   
       genres               106574   4768   
       genres_all           106574   4151   
       information            2349   1586   
       interest           106574.0    NaN   
       language_code         15024     44   
       license              106487    113   
       listens            106574.0    NaN   
       lyricist                311     66   
       number             106574.0    NaN   
       publisher              1263    135   
       tags                 106574   2451   
       title                106573  94986   

                                                                        top  \
album  comments                                                         NaN   
       date_created                                     2015-01-26 13:04:57   
       date_released                                    2008-01-01 00:00:00   
       engineer                                              Ernie Indradat   
       favorites                                                        NaN   
       id                                                               NaN   
       information        <p class="p1" style="margin: 0px; padding: 8px...   
       listens                                                          NaN   
       producer                                                  Joe Belock   
       tags                                                              []   
       title                                              microSong Entries   
       tracks                                                           NaN   
       type                                                           Album   
artist active_year_begin                                2007-01-01 00:00:00   
       active_year_end                                  2016-01-01 00:00:00   
       associated_labels                                       HUSH Records   
       bio                <p><span style="color: #333333; font-family: G...   
       comments                                                         NaN   
       date_created                                     2013-03-31 02:17:41   
       favorites                                               

In [21]:
features_path = os.path.join(fma_metadata_path, 'features.csv')

# load the raw metadata handling the multi-index header
features = pd.read_csv(features_path, index_col=0, header=[0, 1, 2])

#print the shape, sample, and info
print("\n--- Features Shape ---")
print(features.shape)

print("\n--- Features Random Sample (5 rows) ---")
display(features.sample(5, random_state=67))

print("\n--- Features Info ---")
features.info()

print("\n--- Features Description ---")
# Enable pandas to display all rows/columns so nothing gets truncated in the summary
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(features.describe(include='all').T)


--- Features Shape ---
(106574, 518)

--- Features Random Sample (5 rows) ---


feature    chroma_cens                                                    \
statistics    kurtosis                                                     
number              01        02        03        04        05        06   
track_id                                                                   
113262       -1.064637 -1.039122 -1.096008 -0.482389  1.543380 -0.390859   
124022       -0.953732 -1.120171 -0.144379  0.089125 -0.108714 -1.021882   
66876         0.269588  0.951599 -0.022058 -0.436733  0.726334  0.951011   
134564       -1.151917 -1.361734 -1.435525  2.432343 -0.240840 -0.641447   
113665       -0.431926  0.754664 -0.648052 -0.451361 -1.223915 -1.085207   

feature                                             ...   tonnetz            \
statistics                                          ...       std             
number            07        08        09        10  ...        04        05   
track_id                                            ...                       
113262      0.292519  0.407170 -0.155855 -0.207083  ...  0.107896  0.029159   
124022     -0.612001 -1.064214 -1.329795 -1.328080  ...  0.127820  0.038738   
66876       0.986081 -0.353815 -0.596623  0.422824  ...  0.065713  0.016729   
134564     -0.173224 -1.582860 -1.435067 -1.456817  ...  0.222744  0.033984   
113665     -0.777636 -0.779775 -0.799894 -0.722928  ...  0.151268  0.016448   

feature                      zcr                                          \
statistics              kurtosis       max      mean    median       min   
number            06          01        01        01        01        01   
track_id                                                                   
113262      0.019802  334.494537  0.755371  0.025687  0.021484  0.002441   
124022      0.033464  100.238899  0.455566  0.019466  0.012695  0.003906   
66876       0.016954   38.468445  0.372070  0.030276  0.025879  0.001953   
134564      0.032087   31.070511  0.348145  0.029758  0.022461  0.005371   
113665      0.012601   14.641362  0.375000  0.038923  0.025879  0.005859   

feature                          
statistics       skew       std  
number             01        01  
track_id                         
113262      16.058350  0.030535  
124022       9.637485  0.037256  
66876        4.081458  0.021832  
134564       5.113178  0.033310  
113665       3.427355  0.046768  

[5 rows x 518 columns]


--- Features Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 106574 entries, 2 to 155320
Columns: 518 entries, ('chroma_cens', 'kurtosis', '01') to ('zcr', 'std', '01')
dtypes: float64(518)
memory usage: 422.0 MB

--- Features Description ---


count         mean          std  \
feature            statistics number                                       
chroma_cens        kurtosis   01      106574.0     0.439999    32.285524   
                              02      106574.0     0.199756    11.120903   
                              03      106574.0     0.334683     9.980692   
                              04      106574.0     0.270263    19.074557   
                              05      106574.0     0.294806    15.025603   
                              06      106574.0     0.315635    12.176182   
                              07      106574.0     0.273511    16.975991   
                              08      106574.0     0.334927    15.486402   
                              09      106574.0     0.201939     7.364308   
                              10      106574.0     0.285631     8.175257   
                              11      106574.0     0.430549    19.343114   
                              12      106574.0     0.474014    22.967662   
                   max        01      106574.0     0.622666     0.086195   
                              02      106574.0     0.593421     0.083298   
                              03      106574.0     0.610000     0.088985   
                              04      106574.0     0.588934     0.087267   
                              05      106574.0     0.608178     0.089667   
                              06      106574.0     0.595993     0.090984   
                              07      106574.0     0.590399     0.089463   
                              08      106574.0     0.612476     0.089503   
                              09      106574.0     0.590769     0.087707   
                              10      106574.0     0.607276     0.093715   
                              11      106574.0     0.589148     0.092111   
                              12      106574.0     0.598006     0.092947   
                   mean       01      106574.0     0.257083     0.081508   
                              02      106574.0     0.251316     0.073416   
                              03      106574.0     0.249253     0.079626   
                              04      106574.0     0.250905     0.071161   
                              05      106574.0     0.253644     0.078324   
                              06      106574.0     0.244610     0.072895   
                              07      106574.0     0.244378     0.070814   
                              08      106574.0     0.252502     0.078008   
                              09      106574.0     0.251147     0.071696   
                              10      106574.0     0.243101     0.078482   
                              11      106574.0     0.234241     0.072911   
                              12      106574.0     0.232713     0.076043   
                   median     01      106574.0     0.248578     0.095316   
                              02      106574.0     0.245066     0.086288   
                              03      106574.0     0.239654     0.094280   
                              04      106574.0     0.244537     0.083339   
                              05      106574.0     0.245131     0.092062   
                              06      106574.0     0.235755     0.085810   
                              07      106574.0     0.236984     0.083313   
                              08      106574.0     0.243293     0.091651   
                              09      106574.0     0.245229     0.083969   
                              10      106574.0     0.232478     0.092951   
                              11      106574.0     0.225346     0.085580   
                              12      106574.0     0.222565     0.089638   
                   min        01      106574.0     0.009517     0.032707   
                              02      106574.0     0.009110     0.031082   
                              03      106574.0     0.008393     0.029732   
             

####**Analysis of `tracks.csv` and `features.csv` Data Inspection**
  
Check that the row count and columns match what the source says.


Confirm each column was read with the right type: numbers stored as text, dates read as strings, and ID codes read as numbers are common errors.


# **III. Data quality assessment**

# **IV. Data cleaning**
*(with a log of every change)*

# **V. Univariate analysis**

# **VI. Bivariate and multivariate analysis**

# **VII. Type-specific analysis**
*(time, text, images, groups, as applies)*

# **VIII. Summary of findings**

# **IX. Limitations and next steps**

# **X. References**